In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# set project paths

PROJECT_DIR = Path.home() / "Climate-Change-Awareness-Dashboard"
PRECIPITATION_DIR = PROJECT_DIR / "precipitation"

INTERIM_DIR = PRECIPITATION_DIR / "data" / "interim"
OUTPUT_DIR = PRECIPITATION_DIR / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# load temperature dataset

temperature_path = (
    PROJECT_DIR / "data" / "processed_india" /
    "india_monthly_temperature.csv"
)

temperature_df = pd.read_csv(temperature_path)

print("Temperature dataset loaded.")
print("Shape:", temperature_df.shape)
print("Columns:", temperature_df.columns.tolist())

print("\nRainfall interim folder:", INTERIM_DIR)
print("Output folder:", OUTPUT_DIR)

Temperature dataset loaded.
Shape: (322200, 17)
Columns: ['date', 'year', 'month', 'cell_id', 'lat', 'lon', 'tmax_mean', 'tmin_mean', 'tmean_mean', 'tmax_days', 'tmin_days', 'tmax_normal', 'tmin_normal', 'tmean_normal', 'tmax_anom', 'tmin_anom', 'tmean_anom']

Rainfall interim folder: /home/sharanya/Climate-Change-Awareness-Dashboard/precipitation/data/interim
Output folder: /home/sharanya/Climate-Change-Awareness-Dashboard/precipitation/data/processed


In [3]:
import numpy as np
import pandas as pd

# set paths and years

START_YEAR = 1951
END_YEAR = 2025

OUTPUT_DIR = PRECIPITATION_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# select required temperature columns

temperature_columns = [
    "year",
    "month",
    "cell_id",
    "lat",
    "lon",
    "tmax_mean",
    "tmin_mean",
    "tmean_mean",
    "tmax_normal",
    "tmin_normal",
    "tmean_normal",
    "tmax_anom",
    "tmin_anom",
    "tmean_anom"
]

temperature_data = temperature_df[temperature_columns].copy()

# get unique temperature grid locations

temp_cells = (
    temperature_data[["cell_id", "lat", "lon"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# create spatial mapping using the 1951 rainfall grid

first_year_file = INTERIM_DIR / "imd_rainfall_1951.csv"

rain_grid = pd.read_csv(first_year_file)[
    ["latitude", "longitude"]
].drop_duplicates()

mapping_results = []

for _, temp_cell in temp_cells.iterrows():

    t_lat = temp_cell["lat"]
    t_lon = temp_cell["lon"]

    selected_rain = rain_grid[
        (rain_grid["latitude"].between(t_lat - 0.5, t_lat + 0.5)) &
        (rain_grid["longitude"].between(t_lon - 0.5, t_lon + 0.5))
    ].copy()

    lat_diff = np.abs(selected_rain["latitude"] - t_lat)
    lon_diff = np.abs(selected_rain["longitude"] - t_lon)

    is_boundary_lat = np.isclose(lat_diff, 0.5)
    is_boundary_lon = np.isclose(lon_diff, 0.5)

    selected_rain["weight"] = np.where(
        is_boundary_lat & is_boundary_lon, 0.25,
        np.where(is_boundary_lat | is_boundary_lon, 0.5, 1.0)
    )

    selected_rain["lat"] = t_lat
    selected_rain["lon"] = t_lon

    mapping_results.append(selected_rain)

spatial_mapping = pd.concat(mapping_results, ignore_index=True)

print(f"Temperature grid locations: {len(temp_cells)}")
print(f"Spatial mapping records: {len(spatial_mapping)}")

# create monthly records for every year and temperature location

monthly_records = []

for year in range(START_YEAR, END_YEAR + 1):

    print(f"Processing {year}...")

    file_path = INTERIM_DIR / f"imd_rainfall_{year}.csv"

    rainfall = pd.read_csv(file_path)
    rainfall["date"] = pd.to_datetime(rainfall["date"])

    # attach rainfall observations to temperature grid locations

    mapped_rain = rainfall.merge(
        spatial_mapping,
        on=["latitude", "longitude"],
        how="inner"
    )

    # calculate weighted daily rainfall

    mapped_rain = mapped_rain.dropna(subset=["rainfall_mm"])

    mapped_rain["weighted_rain"] = (
        mapped_rain["rainfall_mm"] * mapped_rain["weight"]
    )

    daily = mapped_rain.groupby(
        ["date", "lat", "lon"]
    ).agg(
        weighted_sum=("weighted_rain", "sum"),
        valid_weight=("weight", "sum")
    ).reset_index()

    # calculate weighted average whenever valid weight is positive

    daily["daily_rain"] = np.where(
        daily["valid_weight"] > 0,
        daily["weighted_sum"] / daily["valid_weight"],
        np.nan
    )

    # calculate monthly rainfall

    daily["year"] = year
    daily["month"] = daily["date"].dt.month

    monthly = daily.groupby(
        ["year", "month", "lat", "lon"]
    ).agg(
        monthly_accumulated_rain=("daily_rain", "sum"),
        monthly_mean_rain=("daily_rain", "mean"),
        valid_days=("daily_rain", "count")
    ).reset_index()

    # create all 12 months for every temperature grid location

    months = pd.DataFrame({
        "year": year,
        "month": range(1, 13)
    })

    complete_months = temp_cells.merge(
        months.assign(key=1),
        how="cross"
    )

    # attach calculated monthly rainfall

    monthly = complete_months.merge(
        monthly,
        on=["year", "month", "lat", "lon"],
        how="left",
        validate="one_to_one"
    )

    # fill missing observation counts with zero

    monthly["valid_days"] = monthly["valid_days"].fillna(0).astype(int)

    monthly_records.append(monthly)

# combine all years

all_rainfall = pd.concat(monthly_records, ignore_index=True)

print("\nRainfall consolidation completed.")
print(f"Total rainfall records: {len(all_rainfall)}")

# merge temperature columns

combined_df = all_rainfall.merge(
    temperature_data,
    on=["year", "month", "cell_id", "lat", "lon"],
    how="left",
    validate="one_to_one"
)

# arrange final columns

final_columns = [
    "year",
    "month",
    "cell_id",
    "lat",
    "lon",
    "monthly_accumulated_rain",
    "monthly_mean_rain",
    "valid_days",
    "tmax_mean",
    "tmin_mean",
    "tmean_mean",
    "tmax_normal",
    "tmin_normal",
    "tmean_normal",
    "tmax_anom",
    "tmin_anom",
    "tmean_anom"
]

combined_df = combined_df[final_columns]

# sort and save the dataset

combined_df = combined_df.sort_values(
    ["year", "month", "lat", "lon"]
).reset_index(drop=True)

output_path = OUTPUT_DIR / "01_initial_consolidation.csv"

combined_df.to_csv(output_path, index=False)

print("\nInitial consolidation completed.")
print(f"Total records: {len(combined_df)}")
print(f"Total columns: {len(combined_df.columns)}")
print(f"Saved to: {output_path}")

# inspect columns

print("\n--- columns ---")
print(combined_df.columns.tolist())

# inspect first records

print("\n--- first 5 records ---")
display(combined_df.head())

# inspect middle records

print("\n--- middle 5 records ---")
middle = len(combined_df) // 2
display(combined_df.iloc[middle-2:middle+3])

# inspect last records

print("\n--- last 5 records ---")
display(combined_df.tail())

# inspect missing values

print("\n--- missing values per column ---")
display(combined_df.isna().sum())

Temperature grid locations: 358
Spatial mapping records: 7633
Processing 1951...
Processing 1952...
Processing 1953...
Processing 1954...
Processing 1955...
Processing 1956...
Processing 1957...
Processing 1958...
Processing 1959...
Processing 1960...


KeyboardInterrupt: 

- add location names 
- investigate missing values
- chuck cell_id ?

In [ ]:
# inspect missing values in original temperature dataset

print("Temperature dataset shape:", temperature_df.shape)

print("\nMissing values per column:")
display(temperature_df.isna().sum())

print("\nTotal missing values:", temperature_df.isna().sum().sum())

print("\nMissing value percentage per column:")
display(
    (temperature_df.isna().mean() * 100).round(2)
)

print("\nRows with at least one missing value:")
print(temperature_df.isna().any(axis=1).sum())

print("\nCompletely missing rows:")
print(temperature_df.isna().all(axis=1).sum())

Temperature dataset shape: (322200, 17)

Missing values per column:


date               0
year               0
month              0
cell_id            0
lat                0
lon                0
tmax_mean       7246
tmin_mean       7227
tmean_mean      7251
tmax_days          0
tmin_days          0
tmax_normal      300
tmin_normal      300
tmean_normal     300
tmax_anom       7417
tmin_anom       7398
tmean_anom      7422
dtype: int64


Total missing values: 44861

Missing value percentage per column:


date            0.00
year            0.00
month           0.00
cell_id         0.00
lat             0.00
lon             0.00
tmax_mean       2.25
tmin_mean       2.24
tmean_mean      2.25
tmax_days       0.00
tmin_days       0.00
tmax_normal     0.09
tmin_normal     0.09
tmean_normal    0.09
tmax_anom       2.30
tmin_anom       2.30
tmean_anom      2.30
dtype: float64


Rows with at least one missing value:
7422

Completely missing rows:
0


In [ ]:
import os
from dotenv import load_dotenv

PROJECT_DIR = Path.home() / "Climate-Change-Awareness-Dashboard"

load_dotenv(PROJECT_DIR / ".env")

GEOAPIFY_API_KEY = os.getenv("GEOAPIFY_API_KEY")

if GEOAPIFY_API_KEY:
    print("Geoapify API key loaded successfully.")
else:
    print("API key not found. Check your .env file.")

Geoapify API key loaded successfully.


In [ ]:
import pandas as pd

# load consolidated dataset

consolidated_path = OUTPUT_DIR / "01_initial_consolidation.csv"

combined_df = pd.read_csv(consolidated_path)

# extract unique coordinate pairs

unique_locations = (
    combined_df[["lat", "lon"]]
    .drop_duplicates()
    .sort_values(["lat", "lon"])
    .reset_index(drop=True)
)

print("Consolidated dataset shape:", combined_df.shape)
print("Unique coordinate pairs:", len(unique_locations))

display(unique_locations.head())
display(unique_locations.tail())

Consolidated dataset shape: (322200, 17)
Unique coordinate pairs: 358


,lat,lon
0,8.5,73.5
1,8.5,76.5
2,8.5,77.5
3,8.5,78.5
4,9.5,76.5


,lat,lon
353,35.5,78.5
354,35.5,79.5
355,36.5,73.5
356,36.5,74.5
357,36.5,75.5


In [ ]:
# test a mainland temperature-grid coordinate

test_lat = 12.9716
test_lon = 77.5946

params = {
    "lat": test_lat,
    "lon": test_lon,
    "type": "city",
    "format": "json",
    "lang": "en",
    "apiKey": GEOAPIFY_API_KEY
}

response = requests.get(url, params=params, timeout=30)

if response.status_code == 200:
    result = response.json()

    if result.get("results"):
        place = result["results"][0]

        print("Coordinates:", test_lat, test_lon)
        print("Name:", place.get("name"))
        print("City:", place.get("city"))
        print("State:", place.get("state"))
        print("Country:", place.get("country"))
        print("Distance (km):", round(place.get("distance", 0) / 1000, 2))
    else:
        print("No city result found.")

else:
    print("Request failed:", response.status_code)
    print(response.text)

Coordinates: 12.9716 77.5946
Name: Bengaluru Central City Corporation
City: Bengaluru
State: Karnataka
Country: India
Distance (km): 0.0


In [ ]:
import requests
import pandas as pd

# load unique coordinates from consolidated dataset

unique_locations = (
    combined_df[["lat", "lon"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Total unique coordinates:", len(unique_locations))

# select 10 random coordinates

sample_locations = unique_locations.sample(
    n=10,
    random_state=42
).reset_index(drop=True)

# set api endpoint and threshold

url = "https://api.geoapify.com/v1/geocode/reverse"
MAX_DISTANCE_KM = 25

test_results = []

# test the 10 coordinates

for _, row in sample_locations.iterrows():

    lat = row["lat"]
    lon = row["lon"]

    try:
        # get geographical name and state

        general_response = requests.get(
            url,
            params={
                "lat": lat,
                "lon": lon,
                "format": "json",
                "lang": "en",
                "apiKey": GEOAPIFY_API_KEY
            },
            timeout=30
        )

        general_response.raise_for_status()
        general_results = general_response.json().get("results", [])

        location = None
        state = None

        if general_results:
            location = general_results[0].get("name")
            state = general_results[0].get("state")

        # get nearest city

        city_response = requests.get(
            url,
            params={
                "lat": lat,
                "lon": lon,
                "type": "city",
                "format": "json",
                "lang": "en",
                "apiKey": GEOAPIFY_API_KEY
            },
            timeout=30
        )

        city_response.raise_for_status()
        city_results = city_response.json().get("results", [])

        nearest_city = None

        if city_results:
            city_data = city_results[0]
            city_distance = city_data.get("distance")

            if (
                city_distance is not None
                and city_distance <= MAX_DISTANCE_KM * 1000
            ):
                nearest_city = (
                    city_data.get("city")
                    or city_data.get("name")
                )

        test_results.append({
            "latitude": lat,
            "longitude": lon,
            "location": location,
            "nearest_city": nearest_city,
            "state": state
        })

    except requests.exceptions.RequestException as error:
        print(f"Failed for ({lat}, {lon}): {error}")

        test_results.append({
            "latitude": lat,
            "longitude": lon,
            "location": None,
            "nearest_city": None,
            "state": None
        })

# display test results

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

Total unique coordinates: 358


,latitude,longitude,location,nearest_city,state
0,25.5,86.5,NH31,Khagaria,Bihar
1,14.5,79.5,Kakivaya,Chejerla,Andhra Pradesh
2,27.5,73.5,Fuel Station,Nokha Tehsil,Rajasthan
3,28.5,96.5,Chaglagam Circle,Chaglagam Circle,Arunachal Pradesh
4,16.5,77.5,Subcentre Makthal (a),Makthal mandal,Telangana
5,28.5,91.5,Chigu town,Chigu town,Xizang
6,11.5,72.5,Lakshadweep,Lakshadweep,Lakshadweep
7,16.5,78.5,None,Uppunuthala mandal,Telangana
8,26.5,86.5,None,Laukaha,Bihar
9,21.5,76.5,None,Nepanagar Tahsil,Madhya Pradesh


In [ ]:
import requests
import pandas as pd
import time

# extract unique coordinates from consolidated dataset

unique_locations = (
    combined_df[["lat", "lon"]]
    .drop_duplicates()
    .rename(columns={"lat": "latitude", "lon": "longitude"})
    .reset_index(drop=True)
)

# reuse the 10 previously tested results

location_results = test_results_df.to_dict("records")

completed_coordinates = {
    (row["latitude"], row["longitude"])
    for row in location_results
}

# set api endpoint and threshold

url = "https://api.geoapify.com/v1/geocode/reverse"
MAX_DISTANCE_KM = 25

# process remaining coordinates

for index, row in unique_locations.iterrows():

    lat = row["latitude"]
    lon = row["longitude"]

    if (lat, lon) in completed_coordinates:
        continue

    print(
        f"Processing {len(completed_coordinates) + 1}/"
        f"{len(unique_locations)}: ({lat}, {lon})"
    )

    location = None
    nearest_city = None
    state = None

    try:
        # retrieve exact geographical location and state

        general_response = requests.get(
            url,
            params={
                "lat": lat,
                "lon": lon,
                "format": "json",
                "lang": "en",
                "apiKey": GEOAPIFY_API_KEY
            },
            timeout=30
        )

        general_response.raise_for_status()

        general_results = general_response.json().get("results", [])

        if general_results:
            location_data = general_results[0]
            location = location_data.get("name")
            state = location_data.get("state")

    except requests.exceptions.RequestException as error:
        print(f"Location lookup failed: {error}")

    try:
        # retrieve nearest city within 25 km

        city_response = requests.get(
            url,
            params={
                "lat": lat,
                "lon": lon,
                "type": "city",
                "format": "json",
                "lang": "en",
                "apiKey": GEOAPIFY_API_KEY
            },
            timeout=30
        )

        city_response.raise_for_status()

        city_results = city_response.json().get("results", [])

        if city_results:
            city_data = city_results[0]
            city_distance = city_data.get("distance")

            if (
                city_distance is not None
                and city_distance <= MAX_DISTANCE_KM * 1000
            ):
                nearest_city = (
                    city_data.get("city")
                    or city_data.get("name")
                )

    except requests.exceptions.RequestException as error:
        print(f"Nearest city lookup failed: {error}")

    location_results.append({
        "latitude": lat,
        "longitude": lon,
        "location": location,
        "nearest_city": nearest_city,
        "state": state
    })

    completed_coordinates.add((lat, lon))

    # save progress

    pd.DataFrame(location_results).to_csv(
        OUTPUT_DIR / "location_lookup.csv",
        index=False
    )

    time.sleep(1)

# create final location lookup table

location_lookup_df = pd.DataFrame(location_results)

print("\nLocation lookup completed.")
print("Total locations:", len(location_lookup_df))

display(location_lookup_df.head())
display(location_lookup_df.tail())

Processing 11/358: (8.5, 73.5)
Processing 12/358: (8.5, 76.5)
Processing 13/358: (8.5, 77.5)
Processing 14/358: (8.5, 78.5)
Processing 15/358: (9.5, 76.5)
Processing 16/358: (9.5, 77.5)
Processing 17/358: (9.5, 78.5)
Processing 18/358: (9.5, 79.5)
Processing 19/358: (9.5, 92.5)
Processing 20/358: (10.5, 75.5)
Processing 21/358: (10.5, 76.5)
Processing 22/358: (10.5, 77.5)
Processing 23/358: (10.5, 78.5)
Processing 24/358: (10.5, 79.5)
Processing 25/358: (10.5, 92.5)
Processing 26/358: (11.5, 75.5)
Processing 27/358: (11.5, 76.5)
Processing 28/358: (11.5, 77.5)
Processing 29/358: (11.5, 78.5)
Processing 30/358: (11.5, 79.5)
Processing 31/358: (11.5, 92.5)
Processing 32/358: (12.5, 74.5)
Processing 33/358: (12.5, 75.5)
Processing 34/358: (12.5, 76.5)
Processing 35/358: (12.5, 77.5)
Processing 36/358: (12.5, 78.5)
Processing 37/358: (12.5, 79.5)
Processing 38/358: (12.5, 80.5)
Processing 39/358: (12.5, 92.5)
Processing 40/358: (13.5, 74.5)
Processing 41/358: (13.5, 75.5)
Processing 42/358

,latitude,longitude,location,nearest_city,state
0,25.5,86.5,NH31,Khagaria,Bihar
1,14.5,79.5,Kakivaya,Chejerla,Andhra Pradesh
2,27.5,73.5,Fuel Station,Nokha Tehsil,Rajasthan
3,28.5,96.5,Chaglagam Circle,Chaglagam Circle,Arunachal Pradesh
4,16.5,77.5,Subcentre Makthal (a),Makthal mandal,Telangana


,latitude,longitude,location,nearest_city,state
353,35.5,78.5,Hongliu Town,He'an County,Xinjiang
354,35.5,79.5,Hongliu Town,He'an County,Xinjiang
355,36.5,73.5,Yasin Tehsil,Yasin Tehsil,Gilgit-Baltistan
356,36.5,74.5,Lower Hunza Tehsil,Lower Hunza Tehsil,Gilgit-Baltistan
357,36.5,75.5,Upper Hunza Tehsil,Upper Hunza Tehsil,Gilgit-Baltistan


In [ ]:
# prepare location lookup for merging

location_lookup_df = location_lookup_df.rename(
    columns={
        "latitude": "lat",
        "longitude": "lon"
    }
)

# remove redundant cell_id

combined_df = combined_df.drop(
    columns=["cell_id"],
    errors="ignore"
)

# remove previously added geographical columns if present

combined_df = combined_df.drop(
    columns=["location", "nearest_city", "state"],
    errors="ignore"
)

# merge geographical information

combined_df = combined_df.merge(
    location_lookup_df,
    on=["lat", "lon"],
    how="left",
    validate="many_to_one"
)

# arrange final columns

final_columns = [
    "year",
    "month",
    "lat",
    "lon",
    "location",
    "nearest_city",
    "state",
    "monthly_accumulated_rain",
    "monthly_mean_rain",
    "valid_days",
    "tmax_mean",
    "tmin_mean",
    "tmean_mean",
    "tmax_normal",
    "tmin_normal",
    "tmean_normal",
    "tmax_anom",
    "tmin_anom",
    "tmean_anom"
]

combined_df = combined_df[final_columns]

# save updated dataset

output_path = OUTPUT_DIR / "01_initial_consolidation.csv"

combined_df.to_csv(output_path, index=False)

print("Updated consolidation saved.")
print("Shape:", combined_df.shape)

display(combined_df.head())

Updated consolidation saved.
Shape: (322200, 19)


,year,month,lat,lon,location,nearest_city,state,monthly_accumulated_rain,monthly_mean_rain,valid_days,tmax_mean,tmin_mean,tmean_mean,tmax_normal,tmin_normal,tmean_normal,tmax_anom,tmin_anom,tmean_anom
0,1951,1,8.5,73.5,Laccadive Sea,None,None,NaN,NaN,0,31.165,22.778,26.972,31.784,22.209,26.996,-0.619,0.569,-0.025
1,1951,1,8.5,76.5,Laccadive Sea,None,None,11.604210,0.374329,31,29.918,21.406,25.662,30.252,20.761,25.506,-0.334,0.645,0.156
2,1951,1,8.5,77.5,Mūndrumuduku Kal,Nanguneri,Tamil Nadu,45.013756,1.452057,31,31.051,22.405,26.728,31.325,21.676,26.501,-0.275,0.728,0.227
3,1951,1,8.5,78.5,Gulf of Mannar,None,None,21.057429,0.679272,31,29.852,22.401,26.126,30.257,21.747,26.002,-0.404,0.654,0.125
4,1951,1,9.5,76.5,Neelamperoor Temple Aaratttukulam,Kuttanad,Kerala,16.559665,0.534183,31,30.079,20.919,25.499,30.535,20.346,25.441,-0.456,0.573,0.059


In [ ]:
# inspect missing geographical names

print("Missing location names:", combined_df["location"].isna().sum())
print("Missing nearest cities:", combined_df["nearest_city"].isna().sum())
print("Missing states:", combined_df["state"].isna().sum())

# inspect all unique geographical locations

display(
    location_lookup_df.sort_values(
        ["state", "location"]
    ).reset_index(drop=True)
)

# identify locations without rainfall coverage

missing_rain_locations = (
    combined_df[
        combined_df["monthly_accumulated_rain"].isna()
    ][["lat", "lon", "location", "nearest_city", "state"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nLocations without rainfall coverage:")
display(missing_rain_locations)

Missing location names: 108000
Missing nearest cities: 12600
Missing states: 27000


,lat,lon,location,nearest_city,state
0,16.5,80.5,AB-1,Amaravati,Andhra Pradesh
1,18.5,82.5,Dolaput,Munchingi Puttu,Andhra Pradesh
2,15.5,79.5,Gayamvaripalle,Kanigiri,Andhra Pradesh
3,14.5,79.5,Kakivaya,Chejerla,Andhra Pradesh
4,15.5,78.5,Maddula Eru,Nandyal,Andhra Pradesh
...,...,...,...,...,...
353,21.5,88.5,India,India,None
354,8.5,73.5,Laccadive Sea,None,None
355,8.5,76.5,Laccadive Sea,None,None
356,10.5,75.5,Laccadive Sea,None,None



Locations without rainfall coverage:


,lat,lon,location,nearest_city,state
0,8.5,73.5,Laccadive Sea,None,None
1,9.5,92.5,India,India,None
2,10.5,92.5,India,India,None
3,11.5,72.5,Lakshadweep,Lakshadweep,Lakshadweep
4,11.5,92.5,India,India,None
5,12.5,92.5,India,India,None


In [ ]:
missing_report = (
    df.groupby(["lat", "lon", "location", "nearest_city", "state"])[climate_cols]
    .agg(lambda column: column.isna().sum())
    .reset_index()
)

# retain only locations with at least one missing climate value
missing_report = missing_report[
    missing_report[climate_cols].gt(0).any(axis=1)
].copy()

# count months with missing rainfall values
rainfall_missing = (
    df.groupby(["lat", "lon"])[rainfall_cols]
    .apply(lambda group: group.isna().any(axis=1).sum())
    .rename("rainfall_missing_months")
    .reset_index()
)

# count months with missing temperature values
temperature_missing = (
    df.groupby(["lat", "lon"])[temperature_cols]
    .apply(lambda group: group.isna().any(axis=1).sum())
    .rename("temperature_missing_months")
    .reset_index()
)

# count months with any missing climate value
total_missing = (
    df.groupby(["lat", "lon"])[climate_cols]
    .apply(lambda group: group.isna().any(axis=1).sum())
    .rename("any_missing_months")
    .reset_index()
)

missing_report = missing_report.merge(
    rainfall_missing, on=["lat", "lon"]
).merge(
    temperature_missing, on=["lat", "lon"]
).merge(
    total_missing, on=["lat", "lon"]
)

missing_report["total_months"] = (
    df.groupby(["lat", "lon"]).size().reindex(
        pd.MultiIndex.from_frame(missing_report[["lat", "lon"]])
    ).values
)

missing_report.to_csv(
    OUTPUT_DIR / "locations_with_missing_climate_values.csv",
    index=False
)

print("Locations with missing climate values:", len(missing_report))
print("Report saved successfully.")

Locations with missing climate values: 23
Report saved successfully.


In [ ]:
climate_cols = rainfall_cols + temperature_cols

print("Total rows:", len(df))
print("Total locations:", df[["lat", "lon"]].drop_duplicates().shape[0])

print("\nMissing values per column:")
print(df[climate_cols].isna().sum())

print("\nTotal missing climate values:")
print(df[climate_cols].isna().sum().sum())

print("\nRows with at least one missing climate value:")
print(df[climate_cols].isna().any(axis=1).sum())

print("\nLocations with at least one missing climate value:")
print(
    df.groupby(["lat", "lon"])[climate_cols]
    .apply(lambda group: group.isna().any().any())
    .sum()
)

Total rows: 322200
Total locations: 358

Missing values per column:
monthly_accumulated_rain    5400
monthly_mean_rain           5400
tmax_mean                   7246
tmin_mean                   7227
tmean_mean                  7251
tmax_normal                  300
tmin_normal                  300
tmean_normal                 300
tmax_anom                   7417
tmin_anom                   7398
tmean_anom                  7422
dtype: int64

Total missing climate values:
55661

Rows with at least one missing climate value:
10698

Locations with at least one missing climate value:
30


In [ ]:
# get the 23 affected locations from the missing report
affected_locations = missing_report[["lat", "lon"]].drop_duplicates()

# filter the original dataset to only those locations
df_23 = df.merge(
    affected_locations,
    on=["lat", "lon"],
    how="inner"
)

# count missing values for each climate variable
missing_by_column = df_23[climate_cols].isna().sum()

print("Number of locations:", len(affected_locations))
print("Total rows:", len(df_23))

print("\nMissing values per column across these locations:")
print(missing_by_column)

print("\nTotal missing climate values:", missing_by_column.sum())

print(
    "\nRows with at least one missing climate value:",
    df_23[climate_cols].isna().any(axis=1).sum()
)

Number of locations: 23
Total rows: 20700

Missing values per column across these locations:
monthly_accumulated_rain     900
monthly_mean_rain            900
tmax_mean                   5137
tmin_mean                   5118
tmean_mean                  5142
tmax_normal                  225
tmin_normal                  225
tmean_normal                 225
tmax_anom                   5286
tmin_anom                   5267
tmean_anom                  5291
dtype: int64

Total missing climate values: 33716

Rows with at least one missing climate value: 6191


In [ ]:
# all 30 locations with missing climate values
all_affected = (
    df.groupby(["lat", "lon"])[climate_cols]
    .apply(lambda group: group.isna().any().any())
)

all_affected = all_affected[all_affected].reset_index()[["lat", "lon"]]

# the 23 locations already present in missing_report
reported = missing_report[["lat", "lon"]].drop_duplicates()

# find the locations present in the overall data but absent from the report
missing_seven = all_affected.merge(
    reported,
    on=["lat", "lon"],
    how="left",
    indicator=True
)

missing_seven = missing_seven[
    missing_seven["_merge"] == "left_only"
].drop(columns="_merge")

# retrieve their location details
missing_seven = missing_seven.merge(
    df[["lat", "lon", "location", "nearest_city", "state"]]
    .drop_duplicates(subset=["lat", "lon"]),
    on=["lat", "lon"],
    how="left"
)

display(missing_seven)
print("Number of omitted locations:", len(missing_seven))

,lat,lon,location,nearest_city,state
0,8.5,73.5,Laccadive Sea,NaN,NaN
1,9.5,92.5,India,India,NaN
2,10.5,92.5,India,India,NaN
3,11.5,92.5,India,India,NaN
4,12.5,92.5,India,India,NaN
5,33.5,73.5,NaN,Kallar Syedan Tehsil,Punjab
6,34.5,76.5,NaN,Kargil Tehsil,Ladakh


Number of omitted locations: 7


In [ ]:
# identify missing climate values
climate_missing = df[climate_cols].isna().any(axis=1)

# identify missing or unidentified location information
location_cols = ["location", "nearest_city", "state"]

unidentified_values = {
    "", "unknown", "not identified", "n/a", "na", "none", "null"
}

location_missing = pd.Series(False, index=df.index)

for col in location_cols:
    values = df[col].astype("string").str.strip().str.lower()

    location_missing |= (
        values.isna()
        | values.isin(unidentified_values)
    )

# India alone is not a sufficiently specific location or city
location_missing |= (
    df["location"].astype("string").str.strip().str.lower() == "india"
)

location_missing |= (
    df["nearest_city"].astype("string").str.strip().str.lower() == "india"
)

# combine both conditions
missing_consolidated = df[
    climate_missing | location_missing
].copy()

# save the consolidated report
missing_consolidated.to_csv(
    OUTPUT_DIR / "missing_values_consolidated.csv",
    index=False
)

print("Total rows in original dataset:", len(df))
print("Rows with missing climate values:", climate_missing.sum())
print("Rows with incomplete/unidentified location:", location_missing.sum())
print("Total rows in consolidated report:", len(missing_consolidated))

print("\nReport saved as missing_values_consolidated.csv")

Total rows in original dataset: 322200
Rows with missing climate values: 10698
Rows with incomplete/unidentified location: 135900
Total rows in consolidated report: 142091

Report saved as missing_values_consolidated.csv


In [ ]:
missing_df = pd.read_csv(
    OUTPUT_DIR / "missing_values_consolidated.csv"
)

indian_states = {
    "andhra pradesh", "arunachal pradesh", "assam", "bihar",
    "chhattisgarh", "goa", "gujarat", "haryana",
    "himachal pradesh", "jharkhand", "karnataka", "kerala",
    "madhya pradesh", "maharashtra", "manipur", "meghalaya",
    "mizoram", "nagaland", "odisha", "punjab", "rajasthan",
    "sikkim", "tamil nadu", "telangana", "tripura",
    "uttar pradesh", "uttarakhand", "west bengal",
    "andaman and nicobar islands", "chandigarh",
    "dadra and nagar haveli and daman and diu",
    "delhi", "jammu and kashmir", "ladakh", "lakshadweep",
    "puducherry"
}

state_values = missing_df["state"].astype("string").str.strip().str.lower()

inside_india = state_values.isin(indian_states)
unidentified = state_values.isna() | state_values.isin(
    ["", "unknown", "not identified", "n/a", "na", "none", "null"]
)
outside_india = ~(inside_india | unidentified)

print("Total rows in missing CSV:", len(missing_df))
print("\nInside India:", inside_india.sum())
print("Outside India:", outside_india.sum())
print("Unidentified:", unidentified.sum())

print("\nUnique locations:")
print("Inside India:", missing_df.loc[inside_india, ["lat", "lon"]].drop_duplicates().shape[0])
print("Outside India:", missing_df.loc[outside_india, ["lat", "lon"]].drop_duplicates().shape[0])
print("Unidentified:", missing_df.loc[unidentified, ["lat", "lon"]].drop_duplicates().shape[0])

Total rows in missing CSV: 142091

Inside India: 106954
Outside India: 8137
Unidentified: 27000

Unique locations:
Inside India: 124
Outside India: 20
Unidentified: 30


In [ ]:
# identify rows where every rainfall and temperature value is missing
all_climate_missing = df[climate_cols].isna().all(axis=1)

# remove only those rows
cleaned_df = df.loc[~all_climate_missing].copy()

print("Original rows:", len(df))
print("Rows removed:", all_climate_missing.sum())
print("Remaining rows:", len(cleaned_df))

# save the cleaned main dataset
cleaned_df.to_csv(
    OUTPUT_DIR / "01_initial_consolidation.csv",
    index=False
)

print("\nUpdated 01_initial_consolidation.csv successfully.")

Original rows: 322200
Rows removed: 53
Remaining rows: 322147

Updated 01_initial_consolidation.csv successfully.


In [ ]:
indian_mainland_states = {
    "andhra pradesh", "arunachal pradesh", "assam", "bihar",
    "chhattisgarh", "goa", "gujarat", "haryana",
    "himachal pradesh", "jharkhand", "karnataka", "kerala",
    "madhya pradesh", "maharashtra", "manipur", "meghalaya",
    "mizoram", "nagaland", "odisha", "punjab", "rajasthan",
    "sikkim", "tamil nadu", "telangana", "tripura",
    "uttar pradesh", "uttarakhand", "west bengal",
    "chandigarh", "delhi", "jammu and kashmir", "ladakh",
    "puducherry"
}

state_clean = (
    cleaned_df["state"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# retain only identified Indian mainland states and UTs
keep_mask = state_clean.isin(indian_mainland_states)

# identify locations to retain and omit
retained_df = cleaned_df[keep_mask].copy()
omitted_df = cleaned_df[~keep_mask].copy()

print("Original rows:", len(cleaned_df))
print("Rows retained:", len(retained_df))
print("Rows to omit:", len(omitted_df))

print("\nRetained unique locations:",
      retained_df[["lat", "lon"]].drop_duplicates().shape[0])

print("Omitted unique locations:",
      omitted_df[["lat", "lon"]].drop_duplicates().shape[0])

print("\nStates being omitted:")
print(omitted_df["state"].value_counts(dropna=False))

Original rows: 322147
Rows retained: 254700
Rows to omit: 67447

Retained unique locations: 283
Omitted unique locations: 75

States being omitted:
state
NaN                       26947
Xizang                     9900
Gilgit-Baltistan           5400
Sindh                      4500
Xinjiang                   3600
Sagaing Region             1800
Northern Province           900
Rajshahi Division           900
Chin                        900
Chattogram Division         900
Lakshadweep                 900
Bagamati Province           900
Sylhet Division             900
One                         900
Rangpur Division            900
Trashiyangtse District      900
Tongsa                      900
Thimphu District            900
Kachin State                900
Sudurpashchim Province      900
Lumbini Province            900
Azad Kashmir                900
Khyber Pakhtunkhwa          900
Name: count, dtype: int64


In [ ]:
omitted_locations = (
    omitted_df[
        ["lat", "lon", "location", "nearest_city", "state"]
    ]
    .drop_duplicates()
    .sort_values(["state", "lat", "lon"])
    .reset_index(drop=True)
)

print("Total omitted locations:", len(omitted_locations))

for start in range(0, len(omitted_locations), 15):
    end = min(start + 15, len(omitted_locations))

    print(f"\nLocations {start + 1} to {end}")
    display(omitted_locations.iloc[start:end])

Total omitted locations: 75

Locations 1 to 15


,lat,lon,location,nearest_city,state
0,34.5,73.5,Bachhimar,Muzaffarabad,Azad Kashmir
1,27.5,84.5,Chitwan,Chitwan,Bagamati Province
2,22.5,92.5,Belaichhari Upazila,Belaichhari Upazila,Chattogram Division
3,23.5,93.5,NaN,Tedim Township,Chin
4,35.5,74.5,Goharabad Gah,Diamer District,Gilgit-Baltistan
5,35.5,75.5,Gulabpur Tehsil,Gulabpur Tehsil,Gilgit-Baltistan
6,35.5,76.5,Nayser Brakk,Mashabrum Tehsil,Gilgit-Baltistan
7,36.5,73.5,Yasin Tehsil,Yasin Tehsil,Gilgit-Baltistan
8,36.5,74.5,Lower Hunza Tehsil,Lower Hunza Tehsil,Gilgit-Baltistan
9,36.5,75.5,Upper Hunza Tehsil,Upper Hunza Tehsil,Gilgit-Baltistan



Locations 16 to 30


,lat,lon,location,nearest_city,state
15,26.5,87.5,Drabesh1,Rangeli,One
16,24.5,88.5,Godagari Upazila,Godagari Upazila,Rajshahi Division
17,26.5,88.5,NaN,Debnagar Union,Rangpur Division
18,24.5,94.5,NaN,Tamu Township,Sagaing Region
19,26.5,95.5,Young Kone,Nanyun Township,Sagaing Region
20,24.5,68.5,NaN,Shaheed Fazil Rahu Taluka,Sindh
21,24.5,69.5,Turkiar,Diplo Taluka,Sindh
22,24.5,70.5,Bhaki,Nagarparkar Taluka,Sindh
23,25.5,70.5,Dahli,Dahli Taluka,Sindh
24,27.5,69.5,Khangarh Taluka,Khangarh Taluka,Sindh



Locations 31 to 45


,lat,lon,location,nearest_city,state
30,34.5,79.5,Hongliu Town,He'an County,Xinjiang
31,35.5,77.5,宝玉镇,Cenling County,Xinjiang
32,35.5,78.5,Hongliu Town,He'an County,Xinjiang
33,35.5,79.5,Hongliu Town,He'an County,Xinjiang
34,28.5,88.5,Chig township,Kampa County,Xizang
35,28.5,91.5,Chigu town,Chigu town,Xizang
36,28.5,92.5,Lhuntse Township,Lhuntse Township,Xizang
37,29.5,94.5,Linzhi,Linzhi,Xizang
38,29.5,95.5,Kanas-Dongxing Highway,帮辛乡,Xizang
39,29.5,96.5,米堆河,Yupug township,Xizang



Locations 46 to 60


,lat,lon,location,nearest_city,state
45,8.5,73.5,Laccadive Sea,NaN,NaN
46,8.5,76.5,Laccadive Sea,NaN,NaN
47,8.5,78.5,Gulf of Mannar,NaN,NaN
48,9.5,92.5,India,India,NaN
49,10.5,75.5,Laccadive Sea,NaN,NaN
50,10.5,92.5,India,India,NaN
51,11.5,75.5,India,India,NaN
52,11.5,92.5,India,India,NaN
53,12.5,74.5,Laccadive Sea,NaN,NaN
54,12.5,80.5,Bay of Bengal,NaN,NaN



Locations 61 to 75


,lat,lon,location,nearest_city,state
60,15.5,80.5,India,India,NaN
61,16.5,82.5,India,India,NaN
62,17.5,83.5,Bay of Bengal,NaN,NaN
63,18.5,72.5,Arabian Sea,NaN,NaN
64,18.5,84.5,India,India,NaN
65,19.5,72.5,Arabian Sea,NaN,NaN
66,19.5,85.5,India,India,NaN
67,20.5,70.5,Arabian Sea,NaN,NaN
68,20.5,71.5,India,India,NaN
69,20.5,72.5,Gulf of Khambhat,NaN,NaN


In [ ]:
# use the previously verified retained locations
final_df = retained_df.copy()

print("Rows before filtering:", len(cleaned_df))
print("Rows removed:", len(omitted_df))
print("Rows remaining:", len(final_df))

print("Locations removed:", omitted_df[["lat", "lon"]].drop_duplicates().shape[0])
print("Locations retained:", final_df[["lat", "lon"]].drop_duplicates().shape[0])

# save the updated main dataset
final_df.to_csv(
    OUTPUT_DIR / "01_initial_consolidation.csv",
    index=False
)

print("\nUpdated 01_initial_consolidation.csv successfully.")

Rows before filtering: 322147
Rows removed: 67447
Rows remaining: 254700
Locations removed: 75
Locations retained: 283

Updated 01_initial_consolidation.csv successfully.


In [ ]:
# load the updated main dataset
final_df = pd.read_csv(
    OUTPUT_DIR / "01_initial_consolidation.csv"
)

climate_cols = [
    "monthly_accumulated_rain",
    "monthly_mean_rain",
    "tmax_mean",
    "tmin_mean",
    "tmean_mean",
    "tmax_normal",
    "tmin_normal",
    "tmean_normal",
    "tmax_anom",
    "tmin_anom",
    "tmean_anom"
]

print("Total rows:", len(final_df))
print("Total locations:", final_df[["lat", "lon"]].drop_duplicates().shape[0])

print("\nMissing values per column:")
print(final_df[climate_cols].isna().sum())

print("\nTotal missing climate values:")
print(final_df[climate_cols].isna().sum().sum())

print("\nRows with at least one missing climate value:")
print(final_df[climate_cols].isna().any(axis=1).sum())

print("\nLocations with at least one missing climate value:")
print(
    final_df.groupby(["lat", "lon"])[climate_cols]
    .apply(lambda group: group.isna().any().any())
    .sum()
)

Total rows: 254700
Total locations: 283

Missing values per column:
monthly_accumulated_rain      0
monthly_mean_rain             0
tmax_mean                   761
tmin_mean                   761
tmean_mean                  761
tmax_normal                   0
tmin_normal                   0
tmean_normal                  0
tmax_anom                   761
tmin_anom                   761
tmean_anom                  761
dtype: int64

Total missing climate values:
4566

Rows with at least one missing climate value:
761

Locations with at least one missing climate value:
8


In [ ]:
final_df = pd.read_csv(
    OUTPUT_DIR / "01_initial_consolidation.csv"
)

climate_cols = [
    "monthly_accumulated_rain",
    "monthly_mean_rain",
    "tmax_mean",
    "tmin_mean",
    "tmean_mean",
    "tmax_normal",
    "tmin_normal",
    "tmean_normal",
    "tmax_anom",
    "tmin_anom",
    "tmean_anom"
]

location_cols = ["location", "nearest_city", "state"]

all_check_cols = climate_cols + location_cols

print("Total rows:", len(final_df))
print("Total locations:", final_df[["lat", "lon"]].drop_duplicates().shape[0])

print("\nMissing values per column:")
print(final_df[all_check_cols].isna().sum())

print("\nTotal missing climate values:",
      final_df[climate_cols].isna().sum().sum())

print("Total missing location values:",
      final_df[location_cols].isna().sum().sum())

print("\nRows with at least one missing climate value:",
      final_df[climate_cols].isna().any(axis=1).sum())

print("Rows with at least one missing location value:",
      final_df[location_cols].isna().any(axis=1).sum())

Total rows: 254700
Total locations: 283

Missing values per column:
monthly_accumulated_rain         0
monthly_mean_rain                0
tmax_mean                      761
tmin_mean                      761
tmean_mean                     761
tmax_normal                      0
tmin_normal                      0
tmean_normal                     0
tmax_anom                      761
tmin_anom                      761
tmean_anom                     761
location                    104400
nearest_city                   900
state                            0
dtype: int64

Total missing climate values: 4566
Total missing location values: 105300

Rows with at least one missing climate value: 761
Rows with at least one missing location value: 105300


In [ ]:
# identify rows with any missing temperature value
temperature_cols = [
    "tmax_mean",
    "tmin_mean",
    "tmean_mean",
    "tmax_normal",
    "tmin_normal",
    "tmean_normal",
    "tmax_anom",
    "tmin_anom",
    "tmean_anom"
]

temperature_missing = final_df[temperature_cols].isna().any(axis=1)

# remove those rows
final_df = final_df.loc[~temperature_missing].copy()

print("Rows before removal:", 254700)
print("Rows removed:", temperature_missing.sum())
print("Rows remaining:", len(final_df))
print("Remaining locations:", final_df[["lat", "lon"]].drop_duplicates().shape[0])

# overwrite the main consolidated dataset
final_df.to_csv(
    OUTPUT_DIR / "01_initial_consolidation.csv",
    index=False
)

print("\nUpdated 01_initial_consolidation.csv successfully.")

Rows before removal: 254700
Rows removed: 761
Rows remaining: 253939
Remaining locations: 283

Updated 01_initial_consolidation.csv successfully.


In [ ]:
final_df = pd.read_csv(
    OUTPUT_DIR / "01_initial_consolidation.csv"
)

climate_cols = [
    "monthly_accumulated_rain",
    "monthly_mean_rain",
    "tmax_mean",
    "tmin_mean",
    "tmean_mean",
    "tmax_normal",
    "tmin_normal",
    "tmean_normal",
    "tmax_anom",
    "tmin_anom",
    "tmean_anom"
]

location_cols = ["location", "nearest_city", "state"]

all_check_cols = climate_cols + location_cols

print("Total rows:", len(final_df))
print("Total locations:", final_df[["lat", "lon"]].drop_duplicates().shape[0])

print("\nMissing values per column:")
print(final_df[all_check_cols].isna().sum())

print("\nTotal missing climate values:",
      final_df[climate_cols].isna().sum().sum())

print("Total missing location values:",
      final_df[location_cols].isna().sum().sum())

print("\nRows with at least one missing climate value:",
      final_df[climate_cols].isna().any(axis=1).sum())

print("Rows with at least one missing location value:",
      final_df[location_cols].isna().any(axis=1).sum())

Total rows: 253939
Total locations: 283

Missing values per column:
monthly_accumulated_rain         0
monthly_mean_rain                0
tmax_mean                        0
tmin_mean                        0
tmean_mean                       0
tmax_normal                      0
tmin_normal                      0
tmean_normal                     0
tmax_anom                        0
tmin_anom                        0
tmean_anom                       0
location                    104393
nearest_city                   900
state                            0
dtype: int64

Total missing climate values: 0
Total missing location values: 105293

Rows with at least one missing climate value: 0
Rows with at least one missing location value: 105293


In [ ]:
location_cols = ["location", "nearest_city", "state"]

# count missing location fields in each row
missing_location_count = final_df[location_cols].isna().sum(axis=1)

# remove rows with more than one missing location field
final_df = final_df.loc[missing_location_count <= 1].copy()

print("Remaining rows:", len(final_df))
print(
    "Remaining locations:",
    final_df[["lat", "lon"]].drop_duplicates().shape[0]
)

print("\nMissing location values:")
print(final_df[location_cols].isna().sum())

# save updated dataset
final_df.to_csv(
    OUTPUT_DIR / "01_initial_consolidation.csv",
    index=False
)

print("\nUpdated 01_initial_consolidation.csv successfully.")

Remaining rows: 253939
Remaining locations: 283

Missing location values:
location        104393
nearest_city       900
state                0
dtype: int64

Updated 01_initial_consolidation.csv successfully.


- if location rainfall and temp everything is missing for rows delete those rows.. from the main dataset lets do this now
- okay i want all location names that are chinese.. or like the boundary countries.. pakistan..  sri lanks and even andaman nicobar and lakshadweep such rows to be omitted

In [8]:
missing_city_locations = (
    final_df.loc[
        final_df["nearest_city"].isna(),
        ["lat", "lon", "location", "state"]
    ]
    .drop_duplicates()
)

display(missing_city_locations)

print(
    "Locations without nearest city:",
    len(missing_city_locations)
)

,lat,lon,location,state
92,21.5,72.5,Bharūch,Gujarat


Locations without nearest city: 1
